In [10]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

Note: you may need to restart the kernel to use updated packages.


In [11]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [12]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间与电流处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        current_values = group["Current"]

        # 如果首个测量点 Current == 0，则忽略首点
        if len(group) >= 2 and group["Current"].iloc[0] == 0:
            current_values = current_values.iloc[1:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"]).filter(
        lambda group: not is_bad_current_segment(group)
    ).copy()

    # -----------------------------
    # 7. 每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    selected_indices = []

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        # 用第二个测量点记录
        if len(group) >= 2:
            selected_indices.append(group.index[1])
        else:
            selected_indices.append(group.index[0])

    pulse_sequence = pulse_sequence.loc[selected_indices].reset_index(drop=True)

    # -----------------------------
    # 8. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence


pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

display(pulse_sequence)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.760000+00:00,-1.499470,4.036492,DCH,12_17,DCH/-1.5
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.640000+00:00,-1.498840,3.725950,DCH,12_35,DCH/-1.5
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5
...,...,...,...,...,...,...,...,...,...
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0
258,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.810000+00:00,-1.497851,3.192480,DCH,9_54,DCH/-1.5
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0


In [13]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[~df["Zustand/Current"].isin(["DCH/-1.5"])].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0
...,...,...,...,...,...,...,...,...,...
256,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0


In [14]:
# -----------------------------
# 10. 为当前 filter 后的 pulse 匹配前一个 PAUO 的最后时刻数据
# 输出：
#   1) filtered_pauo：每个 filtered_pulse 前序的最后一个 PAUO 点
#   2) pauo_pulse_table：筛选出的 PAUO 与 filtered_pulse 合并表
# -----------------------------

def build_pauo_pulse_tables(time_diff_sequence, filtered_pulse):
    # 只从完整 time_diff_sequence 里取 PAUO，这样 PAUO 也保留 SOH / SOC / File 等列
    pauo_points = time_diff_sequence[time_diff_sequence["Zustand"].eq("PAUO")].copy()

    pauo_points["pause_end_time"] = pd.to_datetime(pauo_points["Time"], utc=True, errors="coerce").astype("datetime64[ns, UTC]")

    pauo_points = (pauo_points.dropna(subset=["File", "pause_end_time"]).reset_index(drop=True))
    pauo_points["pauo_row"] = np.arange(len(pauo_points))

    pulse_for_match = filtered_pulse.copy().reset_index(drop=True)
    pulse_for_match["pulse_time"] = pd.to_datetime(pulse_for_match["Time"], utc=True, errors="coerce").astype("datetime64[ns, UTC]")
    pulse_for_match["pulse_order"] = np.arange(len(pulse_for_match))

    valid_pulse = pulse_for_match.dropna(subset=["File", "pulse_time"]).copy()

    # 对每一个 filtered_pulse，向前找同一个 File 内时间最近的 PAUO 点
    pause_end_match = pd.merge_asof(
        valid_pulse[["File", "pulse_time", "pulse_order"]]
        .sort_values(["pulse_time", "File"]),

        pauo_points[["File", "pause_end_time", "pauo_row"]]
        .sort_values(["pause_end_time", "File"]),
        left_on="pulse_time",
        right_on="pause_end_time",
        by="File",
        direction="backward"
    )

    pause_end_match = (
        pause_end_match
        .sort_values("pulse_order")
        .reset_index(drop=True)
    )

    pause_end_match["r0_pauo_pulse_dt_s"] = (pause_end_match["pulse_time"] - pause_end_match["pause_end_time"]).dt.total_seconds()

    # 只保留PAUO 点和 pulse 点时间间隔不超过 2 秒的 R0 配对
    pause_end_match = pause_end_match[pause_end_match["r0_pauo_pulse_dt_s"]<=2].copy()

    pause_end_match["match_id"] = np.arange(1, len(pause_end_match) + 1)

    # 表 1：筛选出的 PAUO 点
    matched_pauo_pairs = pause_end_match.dropna(subset=["pauo_row"])[
        ["pulse_order", "pauo_row", "match_id"]
    ].copy()
    
    filtered_pauo = (
        matched_pauo_pairs
        .merge(pauo_points, on="pauo_row", how="inner")
        .drop(columns=["pulse_order", "pauo_row", "pause_end_time"], errors="ignore")
        .reset_index(drop=True)
    )
    filtered_pauo = filtered_pauo[["match_id"] + OUTPUT_COLUMNS ].copy()

    # 表 2：PAUO + PULSE 合并表
    pauo_output = filtered_pauo.copy()
    
    pulse_output = (
        pulse_for_match.merge(
            pause_end_match[["pulse_order", "pauo_row", "match_id"]],
            on="pulse_order",
            how="inner"
        ).drop(columns=["pulse_time", "pulse_order", "pauo_row"], errors="ignore")
    )
    

    common_columns = ["match_id"] + OUTPUT_COLUMNS 

    pauo_pulse_table = pd.concat(
        [
            pauo_output[common_columns],
            pulse_output[common_columns]
        ],
        ignore_index=True,
        sort=False,
    )

    pauo_pulse_table = (
    pauo_pulse_table
    .sort_values(["File", "match_id", "Time"], na_position="last")
    .drop(columns=["match_id"])
    .reset_index(drop=True)
    )

# 单独从表1删除 match_id
    filtered_pauo = (
    filtered_pauo
    .drop(columns=["match_id"])
    .reset_index(drop=True)
    )
    
    return filtered_pauo, pauo_pulse_table

filtered_pauo, pauo_pulse_table = build_pauo_pulse_tables(
    time_diff_sequence,
    filtered_pulse,
)

display(filtered_pauo)
display(pauo_pulse_table)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.750000+00:00,0.0,4.087893,PAUO,12_20,PAUO/0.0
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.890000+00:00,0.0,4.088227,PAUO,12_24,PAUO/0.0
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:38.910000+00:00,0.0,3.776801,PAUO,12_38,PAUO/0.0
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:22.170000+00:00,0.0,3.777579,PAUO,12_42,PAUO/0.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:25.540000+00:00,0.0,3.779025,PAUO,12_46,PAUO/0.0
...,...,...,...,...,...,...,...,...,...
188,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:48.480000+00:00,0.0,3.773688,PAUO,9_43,PAUO/0.0
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:51.800000+00:00,0.0,3.775133,PAUO,9_47,PAUO/0.0
190,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:30.920000+00:00,0.0,3.300823,PAUO,9_57,PAUO/0.0
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.100000+00:00,0.0,3.305715,PAUO,9_61,PAUO/0.0


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.750000+00:00,0.000000,4.087893,PAUO,12_20,PAUO/0.0
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.890000+00:00,0.000000,4.088227,PAUO,12_24,PAUO/0.0
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:38.910000+00:00,0.000000,3.776801,PAUO,12_38,PAUO/0.0
...,...,...,...,...,...,...,...,...,...
381,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5
382,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.100000+00:00,0.000000,3.305715,PAUO,9_61,PAUO/0.0
383,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0
384,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 07:25:17.580000+00:00,0.000000,3.317167,PAUO,9_65,PAUO/0.0


In [15]:
# -----------------------------
# 11. 计算 R0 = delta_Voltage / delta_Current
# -----------------------------
def resistance_r0(pauo_pulse_table):
    r0_table = (
    pauo_pulse_table
    .sort_values(["File", "Time"])
    .reset_index(drop=True)
    .copy()
    )

    id_parts = r0_table["ID"].astype(str).str.rsplit("_", n=1, expand=True)
    
    r0_table["id_number"] = pd.to_numeric(
    id_parts[1],
    errors="coerce"
    )

    id_is_adjacent = (r0_table["File"].eq(r0_table["File"].shift(1))

    & r0_table["id_number"].eq(r0_table["id_number"].shift(1) + 1)
    )

    r0_table["delta_Current"] = (
    r0_table["Current"]- r0_table["Current"].shift(1)
    )

    r0_table["delta_Voltage"] = (
    r0_table["Voltage"]- r0_table["Voltage"].shift(1)
    )
    
    r0_table["R0"] = (r0_table["delta_Voltage"]/ r0_table["delta_Current"] * 1000).abs().where(id_is_adjacent)

    r0_table = (r0_table[r0_table["Zustand"].ne("PAUO")].reset_index(drop=True))

    r0_result = (
    r0_table.drop(columns=["id_number", "delta_Current", "delta_Voltage"])
    .reset_index(drop=True)
    )

    return r0_result

r0_result = resistance_r0(pauo_pulse_table)
display(r0_result)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,26.328217
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,26.230695
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,19.202767
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,19.439036
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0,19.365514
...,...,...,...,...,...,...,...,...,...,...
188,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,18.843590
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,18.792110
190,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,21.145712
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,21.427398


In [16]:
# =============================================================================
# 12. 二阶 RC 拟合：在已计算 R0 的基础上计算 R1 / R2
# =============================================================================
# 模型说明：
#   这里沿用原始数据的电流符号：CHA 为正，DCH 为负。
#   V = OCV + Current * R0 + V1 + V2
#   dV1/dt = -V1/tau1 + Current*R1/tau1
#   dV2/dt = -V2/tau2 + Current*R2/tau2
#
# 拟合窗口：
#   前一个 PAUO 末端点用于初始化 OCV 和 RC 状态；
#   当前 pulse 段的所有采样点参与拟合；
#   当前 pulse 后面的 PAUO 段所有采样点参与拟合。
#
# 输出：
#   R0 / R1 / R2 单位均为 mOhm
#   tau1 / tau2 单位为 s
#   C1 / C2 单位为 F
# =============================================================================
# 二阶 RC 拟合初始值：不同 SOC 使用不同初始值，避免所有 SOC 共用同一组 x0
# R1_fraction 表示动态电阻 r_dyn_guess 中分配给 R1 的比例，剩余比例自动分配给 R2。
RC2_INITIAL_GUESS_BY_SOC = {
    "90%": {"R1_fraction": 0.40, "tau1": 2.0,  "tau2": 150.0},
    "50%": {"R1_fraction": 0.35, "tau1": 4.0,  "tau2": 250.0},
    "10%": {"R1_fraction": 0.35, "tau1": 6.0, "tau2": 300.0}
}

def resistance_r1r2(time_diff_sequence, r0_result):

    seq = time_diff_sequence.copy()
    seq["Zustand_clean"] = (seq["Zustand"].str.strip().str.replace(r"\*+$", "", regex=True))
    seq["Time_dt"] = pd.to_datetime(seq["Time"], utc=True, errors="coerce")

    seq = (
        seq
        .dropna(subset=["Time_dt", "Current", "Voltage"])
        .sort_values(["File", "Time_dt"])
        .reset_index(drop=True)
    )

    # 重新根据 Zustand 连续性划分片段，用来找完整 pulse 段和后续 PAUO 段
    seq["segment_id"] = (seq["File"].ne(seq["File"].shift()) | seq["Zustand_clean"].ne(seq["Zustand_clean"].shift())).cumsum()

    # 用 ID 来定位 pulse 后面的 PAUO：例如 42_17 的后续 pause 只取 42_18[
    seq_id_parts = seq["ID"].astype(str).str.extract(r"^(.*)_(\d+)$")
    seq["id_prefix"] = seq_id_parts[0]
    seq["id_number"] = pd.to_numeric(seq_id_parts[1], errors="coerce")

    r0_table = r0_result.copy()
    r0_table["Time_dt"] = pd.to_datetime(r0_table["Time"], utc=True, errors="coerce")

    def is_pulse_state(s):
        return str(s).startswith(("CHA", "DCH"))

    def simulate_voltage(param, t_s, current_a, ocv_before, pulse_end_s, r0_ohm):
        r1_ohm, r2_ohm, tau1_s, tau2_s, ocv_after = param

        ocv_v = np.where(
            t_s <= pulse_end_s,
            ocv_before + (ocv_after - ocv_before) * (t_s / pulse_end_s),
            ocv_after
        )

        v1 = np.zeros(len(t_s), dtype=float)
        v2 = np.zeros(len(t_s), dtype=float)

        for k in range(1, len(t_s)):
            dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
            a1 = np.exp(-dt / tau1_s)
            a2 = np.exp(-dt / tau2_s)

            v1[k] = a1 * v1[k - 1] + r1_ohm * (1.0 - a1) * current_a[k]
            v2[k] = a2 * v2[k - 1] + r2_ohm * (1.0 - a2) * current_a[k]

        return ocv_v + current_a * r0_ohm + v1 + v2

    result_rows = []

    for _, row in r0_table.iterrows():
        file_name = row["File"]
        pulse_time = row["Time_dt"]
        r0_ohm = float(row["R0"]) / 1000.0

        file_seq = seq[seq["File"].eq(file_name)].copy()
        if file_seq.empty:
            continue

        # 找到 r0_result 对应的 pulse 采样点所在片段
        pulse_candidates = file_seq[(file_seq["Time_dt"].eq(pulse_time)) & (file_seq["Zustand"].map(is_pulse_state))]

        if pulse_candidates.empty:
                continue

        pulse_segment_id = pulse_candidates.iloc[0]["segment_id"]

        pulse_df = file_seq[(file_seq["segment_id"].eq(pulse_segment_id)) & (file_seq["Zustand"].map(is_pulse_state))].copy()
        pulse_df = pulse_df.sort_values("Time_dt").reset_index(drop=True)

        # 如果 pulse 段第一个点 Current == 0，则在 R1/R2 拟合中也去掉该点
        if len(pulse_df) >= 2 and pulse_df["Current"].iloc[0] == 0:
            pulse_df = pulse_df.iloc[1:].reset_index(drop=True)

        if len(pulse_df) < 5:
            continue

        pulse_start_time = pulse_df["Time_dt"].iloc[0]
        pulse_end_time = pulse_df["Time_dt"].iloc[-1]
        # 在每个 pulse 前找前一个 PAUO 末端点
        prev_pauo = file_seq[(file_seq["Time_dt"] < pulse_start_time) & (file_seq["Zustand_clean"].eq("PAUO"))].copy()

        if prev_pauo.empty:
            continue

        prev_pauo_point = prev_pauo.sort_values("Time_dt").iloc[[-1]].copy()

        id_parts = str(row.get("ID", "")).rsplit("_", 1)
        if len(id_parts) != 2:
            continue

        pulse_id_prefix = id_parts[0]
        pulse_id_number = pd.to_numeric(id_parts[1], errors="coerce")
        if pd.isna(pulse_id_number):
            continue

        next_pauo_df = file_seq[
            file_seq["Time_dt"].gt(pulse_end_time)
            & file_seq["Zustand_clean"].eq("PAUO")
            & file_seq["id_prefix"].eq(pulse_id_prefix)
            & file_seq["id_number"].eq(pulse_id_number + 1)
        ].copy()
        next_pauo_df = next_pauo_df.sort_values("Time_dt").reset_index(drop=True)

        if len(next_pauo_df) < 5:
            continue

        soc_label = str(row.get("SOC", "")).strip()
        
        # 脉冲前 PAUO 1 个点 + 脉冲过程所有点 + 脉冲后 PAUO 段所有点（直至下一个脉冲）
        fit_df = pd.concat(
            [prev_pauo_point, pulse_df, next_pauo_df],
            ignore_index=True,
            sort=False
        ).sort_values("Time_dt").reset_index(drop=True)

        t_s = (fit_df["Time_dt"] - fit_df["Time_dt"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)

        current_a = fit_df["Current"].to_numpy(dtype=float)
        voltage_v = fit_df["Voltage"].to_numpy(dtype=float)

        ocv_before = float(prev_pauo_point["Voltage"].iloc[0])
        ocv_after_obs = float(next_pauo_df["Voltage"].iloc[-1])
        ocv_slack = 0.05

        # 从拟合窗口开始，到脉冲结束，一共过了多少秒
        pulse_end_s = (pulse_end_time - fit_df["Time_dt"].iloc[0]).total_seconds()
        pulse_end_s = max(float(pulse_end_s), 1e-9)

        pulse_mask = fit_df["Zustand"].map(is_pulse_state).to_numpy(dtype=bool)
        pauo_mask = fit_df["Zustand_clean"].eq("PAUO").to_numpy(dtype=bool)
        prev_point_mask = fit_df["Time_dt"].eq(prev_pauo_point["Time_dt"].iloc[0]).to_numpy(dtype=bool)

        # 前一个 PAUO 末端点只用于初始化，不参与误差计算
        fit_mask = ~prev_point_mask
        pulse_fit_mask = fit_mask & pulse_mask
        pauo_fit_mask = fit_mask & pauo_mask

        n_pulse = int(pulse_fit_mask.sum())
        n_pauo = int(pauo_fit_mask.sum())

        if n_pulse == 0 or n_pauo == 0:
            continue

        # 用时间跨度归一化权重，而不是用点数归一化
        pulse_duration_s = (pulse_df["Time_dt"].iloc[-1] - pulse_df["Time_dt"].iloc[0]).total_seconds()
        pauo_duration_s = (next_pauo_df["Time_dt"].iloc[-1] - next_pauo_df["Time_dt"].iloc[0]).total_seconds()

        # 设置 pulse 段和 PAUO 段在拟合中的总权重占比
        pulse_weight_total = 0.6
        pauo_weight_total = 0.4

        pulse_weight_per_s = pulse_weight_total / max(pulse_duration_s, 1.0)
        pauo_weight_per_s = pauo_weight_total / max(pauo_duration_s, 1.0)

        weights = np.zeros(len(fit_df), dtype=float)

        t_s_full = (fit_df["Time_dt"] - fit_df["Time_dt"].iloc[0]).dt.total_seconds().to_numpy(dtype=float)

        for k in range(1, len(fit_df)):
            dt_k = max(t_s_full[k] - t_s_full[k - 1], 0.0)

            if pulse_fit_mask[k]:
                weights[k] = np.sqrt(pulse_weight_per_s * dt_k)
            elif pauo_fit_mask[k]:
                weights[k] = np.sqrt(pauo_weight_per_s * dt_k)

        # 初值：先用 pulse 末端电压粗略估计动态电阻；
        # 再按当前 SOC 选择 R1/R2 分配比例和 tau 初值，避免所有 SOC 共用同一组 x0。
        i_pulse_mean = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))
        v_pulse_tail = float(np.nanmedian(pulse_df["Voltage"].tail(min(5, len(pulse_df))).to_numpy(dtype=float)))
        i_pulse_abs = float(np.nanmedian(np.abs(pulse_df["Current"].to_numpy(dtype=float))))

        if not np.isfinite(i_pulse_abs) or i_pulse_abs <= 1e-9:
            continue

        r_dyn_guess = abs((ocv_after_obs - v_pulse_tail) / i_pulse_mean) - r0_ohm
        r_dyn_guess = float(np.clip(r_dyn_guess, 0.002, 0.08))

        soc_initial = RC2_INITIAL_GUESS_BY_SOC[soc_label]

        r1_fraction = float(soc_initial["R1_fraction"])
        r1_fraction = float(np.clip(r1_fraction, 0.05, 0.95))
        r2_fraction = 1.0 - r1_fraction

        x0 = np.array([
            r1_fraction * r_dyn_guess,       # R1, Ohm；按当前 SOC 设置比例
            r2_fraction * r_dyn_guess,       # R2, Ohm；剩余比例分配给 R2
            float(soc_initial["tau1"]),      # tau1, s；按当前 SOC 设置
            float(soc_initial["tau2"]),      # tau2, s；按当前 SOC 设置
            ocv_after_obs
        ], dtype=float)

        # 在 resistance_r1r2 函数内，判断SOC和Zustand后分别设定bound：
        is_dch = str(row.get("Zustand", "")).startswith("DCH")
        is_cha = str(row.get("Zustand", "")).startswith("CHA")
 
        ocv_lower = ocv_after_obs - ocv_slack
        ocv_upper = ocv_after_obs + ocv_slack
 
        if is_cha:
            ocv_lower = max(ocv_lower, ocv_before)
        elif is_dch:
            ocv_upper = min(ocv_upper, ocv_before)

        if soc_label == "10%":
            if is_dch:
                lower = np.array([1e-6, 1e-6, 0.5,  80.0, ocv_lower], dtype=float)
                upper = np.array([0.10,  0.8, 30.0, 600.0, ocv_upper], dtype=float)
            else:  # CHA
                tau2_lower = 40.0 if i_pulse_abs < 2.0 else 80.0
                lower = np.array([1e-6, 1e-6, 0.5,  tau2_lower, ocv_lower], dtype=float)
                upper = np.array([0.08,  0.06, 30.0, 600.0, ocv_upper], dtype=float)
 
        elif soc_label == "50%":
            tau2_lower = 30.0 if (is_cha and i_pulse_abs < 2.0) else 60.0
            lower = np.array([1e-6, 0.003, 0.5,  tau2_lower, ocv_lower], dtype=float)
            upper = np.array([0.08,  0.05,  25.0, 700.0, ocv_upper], dtype=float)
 
        else:  # 90%
            lower = np.array([1e-6, 1e-6, 0.5,  20.0, ocv_lower], dtype=float)
            upper = np.array([0.06,  0.05,  15.0, 600.0, ocv_upper], dtype=float)
        
        # 根据当前 PAUO 段长度动态限制 tau2 上限, tau2 上限 = min(当前条件原本的 tau2 上限, 0.5 * PAUO 时长)
        tau2_upper_dyn = min(upper[3], 0.5 * pauo_duration_s)

        # 防止 PAUO 过短时 tau2 上限过低
        tau2_upper_dyn = max(tau2_upper_dyn, 100.0)

        # 保证 tau2 上界一定大于下界
        tau2_upper_dyn = max(tau2_upper_dyn, lower[3] + 1.0)

        upper[3] = tau2_upper_dyn
           
        # x0 赋值之后，对 10% DCH 单独覆盖
        if soc_label == "10%" and is_dch:
            x0 = np.array([
            0.25 * r_dyn_guess,   # R1 占比更小，让 R2 有更大空间
            0.75 * r_dyn_guess,   # R2 占大头
            6.0,                   # tau1 初始值往大调，避免 branch swapping
            300.0,                 # tau2 初始值
            ocv_after_obs
        ], dtype=float)
        x0 = np.clip(x0, lower + 1e-9, upper - 1e-9)

        def residual(param):
            voltage_hat = simulate_voltage(
                param,
                t_s,
                current_a,
                ocv_before,
                pulse_end_s,
                r0_ohm
            )
            return (voltage_hat - voltage_v) * weights

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=5000
            )

            r1_ohm, r2_ohm, tau1_s, tau2_s, _ = fit.x

            voltage_fit = simulate_voltage(
                fit.x,
                t_s,
                current_a,
                ocv_before,
                pulse_end_s,
                r0_ohm
            )

            err_mv = (voltage_fit[fit_mask] - voltage_v[fit_mask]) * 1000.0
            rmse_mv = float(np.sqrt(np.mean(err_mv ** 2)))

            y_true = voltage_v[fit_mask]
            y_pred = voltage_fit[fit_mask]

            ss_res = np.sum((y_true - y_pred) ** 2)
            ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

            if ss_tot > 0:
                r2_score = float(1.0 - ss_res / ss_tot)
            else:
                r2_score = np.nan

            c1_f = tau1_s / r1_ohm
            c2_f = tau2_s / r2_ohm

            output_row = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": r2_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": tau2_s,
                "C1": c1_f,
                "C2": c2_f,
                "RMSE_mV": rmse_mv,
                "R2_score": r2_score
            })
            result_rows.append(output_row)

        except Exception:
            output_row = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            output_row.update({
                "R1": np.nan,
                "R2": np.nan,
                "tau1": np.nan,
                "tau2": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "RMSE_mV": np.nan,
                "R2_score": np.nan
            })
            result_rows.append(output_row)

    return pd.DataFrame(result_rows)


r1r2_result = resistance_r1r2(time_diff_sequence, r0_result)
display(r1r2_result)

r2_summary = (
    r1r2_result
    .groupby("SOC", as_index=False)
    .agg({
        "RMSE_mV": "mean",
        "R2_score": "mean"
    })
)

display(r2_summary)

,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R1,R2,tau1,tau2,C1,C2,RMSE_mV,R2_score
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,26.328217,10.448090,13.958310,8.752094,80.773338,837.673986,5786.756404,0.181438,0.999617
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,26.230695,10.778998,17.116425,9.184946,124.908962,852.115005,7297.607982,0.460908,0.999374
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,19.202767,8.538489,11.486803,11.274542,76.129441,1320.437675,6627.557033,0.113974,0.999664
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,19.439036,10.083146,31.106603,13.456847,228.645889,1334.588147,7350.397185,0.344611,0.999286
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0,19.365514,8.634190,17.738463,10.922570,141.634344,1265.036985,7984.589529,0.266898,0.999591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,18.843590,9.776999,30.432863,13.094918,224.586621,1339.359668,7379.740250,0.351423,0.999244
189,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,18.792110,8.297475,17.391101,10.625284,132.911473,1280.544296,7642.499104,0.257455,0.999613
190,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,21.145712,12.647088,6.994023,17.067083,77.255826,1349.487229,11045.977828,0.269454,0.997860
191,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,21.427398,18.400564,140.538604,24.381715,600.000000,1325.052592,4269.289610,0.805806,0.996382


,SOC,RMSE_mV,R2_score
0,10%,0.558880,0.997450
1,50%,0.271421,0.999554
2,90%,0.614299,0.998874


In [17]:
# =============================================================================
# 13. 手动删除核查后的异常拟合结果
# =============================================================================
# 该点已核查：90% SOC / CHA 1.5A / SOH = 78.2%
# pulse 起始阶段存在约 0.02 V 快速电压波动，会带来大约 13mOhm 的电阻扰动，影响 R1/R2 辨识。
# 这里只删除 r1r2_result 中对应的一行结果，不改变原始数据，也不重新拟合。

manual_soh = pd.to_numeric(r1r2_result["SOH"], errors="coerce").round(1)
manual_current_abs = pd.to_numeric(r1r2_result["Current"], errors="coerce").abs().round(1)

remove_manual_mask = (
    r1r2_result["SOC"].astype(str).eq("90%")
    & r1r2_result["Zustand"].astype(str).str.startswith("CHA")
    & manual_current_abs.eq(1.5)
    & manual_soh.eq(78.2)
)

r1r2_result = r1r2_result.loc[~remove_manual_mask].reset_index(drop=True)

display(r1r2_result)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R1,R2,tau1,tau2,C1,C2,RMSE_mV,R2_score
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,26.328217,10.448090,13.958310,8.752094,80.773338,837.673986,5786.756404,0.181438,0.999617
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,26.230695,10.778998,17.116425,9.184946,124.908962,852.115005,7297.607982,0.460908,0.999374
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,19.202767,8.538489,11.486803,11.274542,76.129441,1320.437675,6627.557033,0.113974,0.999664
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,19.439036,10.083146,31.106603,13.456847,228.645889,1334.588147,7350.397185,0.344611,0.999286
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.570000+00:00,2.996974,3.837063,CHA,12_47,CHA/3.0,19.365514,8.634190,17.738463,10.922570,141.634344,1265.036985,7984.589529,0.266898,0.999591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,18.843590,9.776999,30.432863,13.094918,224.586621,1339.359668,7379.740250,0.351423,0.999244
188,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,18.792110,8.297475,17.391101,10.625284,132.911473,1280.544296,7642.499104,0.257455,0.999613
189,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,21.145712,12.647088,6.994023,17.067083,77.255826,1349.487229,11045.977828,0.269454,0.997860
190,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,21.427398,18.400564,140.538604,24.381715,600.000000,1325.052592,4269.289610,0.805806,0.996382


In [18]:
soh_r_table = r1r2_result.copy()

# 提取 CHA / DCH 信息
soh_r_table["Pulse_type"] = (
    soh_r_table["Zustand"]
    .astype(str)
    .str.extract(r"^(CHA|DCH)", expand=False)
)

# 电流取绝对值，避免 DCH 显示成负数
soh_r_table["Current_abs"] = soh_r_table["Current"].abs()

# 生成图例标签，例如：90% CHA 3.0A
soh_r_table["Curve_label"] = (
    soh_r_table["SOC"].astype(str)
    + " "
    + soh_r_table["Pulse_type"].astype(str)
    + " "
    + soh_r_table["Current_abs"].round(1).astype(str)
    + "A"
)

# 排序
soh_r_table = soh_r_table.sort_values(["SOH", "SOC", "Pulse_type", "Current_abs"])

# 先显示宽表：每个 pulse 一行，R0/R1/R2 在同一行
display(soh_r_table[["SOH", "SOC", "Zustand", "Pulse_type", "Current", "Current_abs", "R0", "R1", "R2"]])

# 分别画 R0, R1, R2 三张图
# 统一 SOC 格式：把 "10%" / 10 都转成数值 10
soh_r_table["SOC_num"] = (
    soh_r_table["SOC"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
    .round()
    .astype(int)
)

# 每个小图里的三条曲线
soh_r_table["Pulse_label"] = (
    soh_r_table["Pulse_type"].astype(str)
    + " "
    + soh_r_table["Current_abs"].round(1).astype(str)
    + "A"
)

soc_order = [10, 50, 90]
pulse_order = ["CHA 1.5A", "CHA 3.0A", "DCH 3.0A"]

# 给9条线固定颜色
color_map = {
    "10% CHA 1.5A": "#636EFA",
    "10% CHA 3.0A": "#EF553B",
    "10% DCH 3.0A": "#00CC96",
    "50% CHA 1.5A": "#AB63FA",
    "50% CHA 3.0A": "#FFA15A",
    "50% DCH 3.0A": "#19D3F3",
    "90% CHA 1.5A": "#FF6692",
    "90% CHA 3.0A": "#FF97FF",
    "90% DCH 3.0A": "#B6E880",
}

for resistance_name in ["R0", "R1", "R2"]:

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[f"{soc}% SOC" for soc in soc_order],
        shared_yaxes=False,
        horizontal_spacing=0.06
    )

    for col, soc in enumerate(soc_order, start=1):
        for pulse in pulse_order:
            label = f"{soc}% {pulse}"

            d = soh_r_table[
                (soh_r_table["SOC_num"] == soc)
                & (soh_r_table["Pulse_label"] == pulse)
            ].sort_values("SOH")

            if d.empty:
                continue

            fig.add_trace(
                go.Scatter(
                    x=d["SOH"],
                    y=d[resistance_name],
                    mode="lines+markers",
                    name=label,
                    legendgroup=label,
                    line=dict(color=color_map[label], width=2),
                    marker=dict(color=color_map[label], size=6),
                ),
                row=1,
                col=col
            )

    fig.update_layout(
        template="plotly_white",
        title=f"SOH vs SOC 10%, 50%, 90% at {resistance_name}",
        width=1250,
        height=500,
        legend_title_text="SOC / pulse / current"
    )

    for col in range(1, 4):
        fig.update_xaxes(autorange="reversed", row=1, col=col)

    fig.update_xaxes(title_text="SOH (%)", row=1, col=2)
    for col in range(1, 4):
        fig.update_yaxes(title_text=f"{resistance_name} (mOhm)", row=1, col=col)

    fig.show()

,SOH,SOC,Zustand,Pulse_type,Current,Current_abs,R0,R1,R2
180,77.5,10%,CHA,CHA,1.498795,1.498795,30.933861,16.067183,1.735443
182,77.5,10%,CHA,CHA,2.997513,2.997513,30.860536,14.928945,26.574542
181,77.5,10%,DCH,DCH,-2.997885,2.997885,31.280828,20.166952,131.468429
177,77.5,50%,CHA,CHA,1.497895,1.497895,27.018469,9.329855,14.654857
179,77.5,50%,CHA,CHA,2.998592,2.998592,27.549464,10.422301,20.694986
...,...,...,...,...,...,...,...,...,...
156,96.2,50%,CHA,CHA,2.999671,2.999671,18.310264,7.704851,15.300600
155,96.2,50%,DCH,DCH,-2.999864,2.999864,18.211324,9.388027,27.010939
151,96.2,90%,CHA,CHA,1.499424,1.499424,21.207321,7.970383,15.904355
153,96.2,90%,CHA,CHA,2.998952,2.998952,21.317725,8.302361,15.640593
